# Task 4 — Join and Transformation

Join **validated** KAUST, KFUPM, and KSU records, apply the shared technology filter, produce the audit files, and save `data/processed/final.csv`.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from main import apply_team_technology_filter
from src import transform
from src.config import get_paths, load_config
from src.schema import SCHEMA_COLUMNS

config = load_config(ROOT)
paths = get_paths(config).ensure()


In [2]:
validated = [
    pd.read_csv(paths.interim / "KAUST_validated.csv"),
    pd.read_csv(paths.interim / "KFUPM_validated.csv"),
    pd.read_csv(paths.interim / "KSU_validated.csv"),
]

combined = transform.combine_sources(validated)
print("Joined validated records:", len(combined))
print(combined["university"].value_counts())


Joined validated records: 16975
university
KSU      15622
KAUST      938
KFUPM      415
Name: count, dtype: int64


## Technology filter

KAUST/KFUPM are matched on `title + abstract`. KSU is matched with the same whole-word rules on its raw `Article Title + Author Keywords`, then those decisions are joined back by stable `research_id`. The actual filtering happens only after schema validation.

In [3]:
selected, excluded, review = apply_team_technology_filter(combined, config, paths)

print("Technology candidates:", len(selected))
print("No keyword match:", len(excluded))
print(selected["university"].value_counts())


Technology candidates: 1817
No keyword match: 15158
university
KSU      1483
KFUPM     211
KAUST     123
Name: count, dtype: int64


## Transformation rules

In [4]:
transform.transformation_rules()

,rule_id,description,input_columns,output_column
0,R1,Convert publication_year to a nullable integer,publication_year,publication_year
1,R2,Flag whether a DOI is available,doi,has_doi
2,R3,Count the words in each abstract,abstract,abstract_word_count
3,R4,Keep records matching a technology keyword usi...,"title, abstract, source-specific author keywords",filtered rows


In [5]:
final = transform.apply_transformations(selected)
shared = transform.shared_doi_report(final)

filter_dir = paths.processed / "technology_filter"
filter_dir.mkdir(parents=True, exist_ok=True)

final.to_csv(paths.processed / "final.csv", index=False)
shared.to_csv(paths.processed / "shared_doi_review.csv", index=False)
review.to_csv(filter_dir / "filter_review.csv", index=False)
excluded.to_csv(filter_dir / "no_keyword_match.csv", index=False)

print("Saved final.csv:", final.shape)
print("Shared DOI values in final:", shared["doi"].nunique())


Saved final.csv: (1817, 15)
Shared DOI values in final: 2


In [6]:
assert final["research_id"].is_unique
assert final["doi"].notna().all()
assert final["url"].notna().all()
assert len(final) + len(excluded) == len(combined)
print("Task 4 checks passed.")


Task 4 checks passed.
